In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
from pysus import sim


In [2]:
codigos_x = [f"X{i}" for i in range(85, 100)]
codigo_y = [f"Y0{i}" for i in range(0, 10)]
codigo_agressao = codigos_x + codigo_y


In [ ]:
estados = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", 
        "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", 
        "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]

anos = list(range(2015, 2022))

if os.path.exists("violencia_feminina.parquet"):    
    violencia_feminina = pd.read_parquet("violencia_feminina.parquet")
    print("Dados carregados do arquivo parquet.")
else:
    dfs = []
    
    for estado in estados:
        for ano in anos:
            df_temp = sim(state=estado, year=ano)
            df_filtrado = df_temp[
                (df_temp["SEXO"] == "2") &
                (df_temp["CAUSABAS"].str[:3].isin(codigo_agressao))
            ]
            dfs.append(df_filtrado)
            del df_temp
                
    violencia_feminina = pd.concat(dfs, ignore_index=True)
    violencia_feminina.to_parquet("violencia_feminina.parquet")

violencia_feminina["ANO"] = pd.to_datetime(
violencia_feminina["DTOBITO"], format="%d%m%Y", errors="coerce"
    ).dt.year

print(f"Total de registros: {violencia_feminina.shape[0]}")
print(f"total de colunas: {violencia_feminina.shape[1]}")


Dados carregados do arquivo parquet.
Total de registros: 33873
total de colunas: 88


In [4]:
violencia_feminina["ESTADO"] = violencia_feminina["CODMUNOCOR"].str.strip().str[:2]

codigo_estado = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF"
}

violencia_feminina["ESTADO"] = violencia_feminina["ESTADO"].map(codigo_estado)


In [5]:
print(violencia_feminina["ESTADO"].head(5))


0    AC
1    AC
2    AC
3    AC
4    AC
Name: ESTADO, dtype: object


In [6]:
violencia_feminina["ESTADO"].value_counts()


ESTADO
SP    3149
BA    3024
RJ    2361
MG    2339
CE    2194
PA    1832
RS    1826
PE    1742
PR    1521
GO    1389
MA     936
ES     812
AM     811
SC     713
RN     687
MT     682
PB     637
AL     624
MS     461
PI     390
DF     384
SE     376
RO     367
TO     277
RR     223
AC     197
AP     131
Name: count, dtype: int64

In [7]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2015|2016|2017|2018|2019|2020|2021|2022/variaveis/9324?localidades=N3[all]"

response = requests.get(url)
dados = response.json()
print(response.status_code)

200


In [8]:
registros = []

for serie in dados[0]["resultados"][0]["series"]:
    estado = serie["localidade"]["nome"]
    for ano, populacao in serie["serie"].items():
        registros.append({
            "ESTADO_NOME": estado,
            "ANO": int(ano),
            "POPULACAO": int(populacao)
        })
        

populacao_df = pd.DataFrame(registros)
print(populacao_df.head(5))
print(populacao_df.shape)


  ESTADO_NOME   ANO  POPULACAO
0    Rondônia  2015    1768204
1    Rondônia  2016    1787279
2    Rondônia  2017    1805788
3    Rondônia  2018    1757589
4    Rondônia  2019    1777225
(189, 3)


In [9]:
url = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2015|2016|2017|2018|2019|2020|2021|2022/variaveis/9324?localidades=N3[all]"

response = requests.get(url)
dados = response.json()


In [10]:
registros = []

for serie in dados[0]['resultados'][0]['series']:
    estado = serie['localidade']['nome']
    for ano, populacao in serie['serie'].items():
        registros.append({
            'ESTADO_NOME': estado,
            'ANO': int(ano),
            'POPULACAO': int(populacao)
        })

populacao_df = pd.DataFrame(registros)
print(populacao_df.head(10))
print(populacao_df.shape)

  ESTADO_NOME   ANO  POPULACAO
0    Rondônia  2015    1768204
1    Rondônia  2016    1787279
2    Rondônia  2017    1805788
3    Rondônia  2018    1757589
4    Rondônia  2019    1777225
5    Rondônia  2020    1796460
6    Rondônia  2021    1815278
7        Acre  2015     803513
8        Acre  2016     816687
9        Acre  2017     829619
(189, 3)


In [11]:
import json

# Imprime o JSON formatado e legível
print(json.dumps(dados[0]["resultados"][0]["series"][0], indent=4))


{
    "localidade": {
        "id": "11",
        "nivel": {
            "id": "N3",
            "nome": "Unidade da Federa\u00e7\u00e3o"
        },
        "nome": "Rond\u00f4nia"
    },
    "serie": {
        "2015": "1768204",
        "2016": "1787279",
        "2017": "1805788",
        "2018": "1757589",
        "2019": "1777225",
        "2020": "1796460",
        "2021": "1815278"
    }
}


In [12]:
# Extrair o id da localidade e mapear para sigla
id_estado = {}
for serie in dados[0]["resultados"][0]["series"]:
    id_estado[serie["localidade"]["id"]] = serie["localidade"]["nome"]

populacao_df["ESTADO"] = populacao_df["ESTADO_NOME"].map(
    {v: codigo_estado[k] for k, v in id_estado.items()}
)

print(populacao_df.head(10))


  ESTADO_NOME   ANO  POPULACAO ESTADO
0    Rondônia  2015    1768204     RO
1    Rondônia  2016    1787279     RO
2    Rondônia  2017    1805788     RO
3    Rondônia  2018    1757589     RO
4    Rondônia  2019    1777225     RO
5    Rondônia  2020    1796460     RO
6    Rondônia  2021    1815278     RO
7        Acre  2015     803513     AC
8        Acre  2016     816687     AC
9        Acre  2017     829619     AC


In [13]:
violencia_feminina["ANO"] = pd.to_datetime(
    violencia_feminina["DTOBITO"], format="%d%m%Y", errors="coerce"
).dt.year

print(violencia_feminina[["DTOBITO", "ANO"]].head(10))


    DTOBITO   ANO
0  13022015  2015
1  10012015  2015
2  29032015  2015
3  20012015  2015
4  02032015  2015
5  22102015  2015
6  13052015  2015
7  07062015  2015
8  16062015  2015
9  24062015  2015


In [14]:
import json
print(json.dumps(dados[0]["resultados"][0]["series"][0], indent=2))

{
  "localidade": {
    "id": "11",
    "nivel": {
      "id": "N3",
      "nome": "Unidade da Federa\u00e7\u00e3o"
    },
    "nome": "Rond\u00f4nia"
  },
  "serie": {
    "2015": "1768204",
    "2016": "1787279",
    "2017": "1805788",
    "2018": "1757589",
    "2019": "1777225",
    "2020": "1796460",
    "2021": "1815278"
  }
}


In [15]:
print(violencia_feminina.columns.tolist())

['CONTADOR', 'ORIGEM', 'TIPOBITO', 'DTOBITO', 'HORAOBITO', 'NATURAL', 'CODMUNNATU', 'DTNASC', 'IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC', 'ESC2010', 'SERIESCFAL', 'OCUP', 'CODMUNRES', 'LOCOCOR', 'CODESTAB', 'ESTABDESCR', 'CODMUNOCOR', 'IDADEMAE', 'ESCMAE', 'ESCMAE2010', 'SERIESCMAE', 'OCUPMAE', 'QTDFILVIVO', 'QTDFILMORT', 'GRAVIDEZ', 'SEMAGESTAC', 'GESTACAO', 'PARTO', 'OBITOPARTO', 'PESO', 'TPMORTEOCO', 'OBITOGRAV', 'OBITOPUERP', 'ASSISTMED', 'EXAME', 'CIRURGIA', 'NECROPSIA', 'LINHAA', 'LINHAB', 'LINHAC', 'LINHAD', 'LINHAII', 'CAUSABAS', 'CB_PRE', 'CRM', 'COMUNSVOIM', 'DTATESTADO', 'CIRCOBITO', 'ACIDTRAB', 'FONTE', 'NUMEROLOTE', 'TPPOS', 'DTINVESTIG', 'CAUSABAS_O', 'DTCADASTRO', 'ATESTANTE', 'STCODIFICA', 'CODIFICADO', 'VERSAOSIST', 'VERSAOSCB', 'FONTEINV', 'DTRECEBIM', 'ATESTADO', 'DTRECORIGA', 'CAUSAMAT', 'ESCMAEAGR1', 'ESCFALAGR1', 'STDOEPIDEM', 'STDONOVA', 'DIFDATA', 'NUDIASOBCO', 'NUDIASOBIN', 'DTCADINV', 'TPOBITOCOR', 'DTCONINV', 'FONTES', 'TPRESGINFO', 'TPNIVELINV', 'NUDIASINF'

In [16]:
print(violencia_feminina["ANO"].head())

0    2015
1    2015
2    2015
3    2015
4    2015
Name: ANO, dtype: int32


In [18]:
print(violencia_feminina["DTOBITO"].tail(5))


33838    24082021
33839    11112021
33840    04112021
33841    24072021
33842    27092021
Name: DTOBITO, dtype: object
